# vLLM trên Colab — Setup & Serve

Notebook này clone repo dev từ GitHub, chạy script setup (CUDA/torch/vllm) rồi khởi động vLLM OpenAI-compatible server ở background, cuối cùng gửi thử một request để kiểm tra.

Lưu ý: Colab (T4/L4/A100) chỉ dùng để dev/thử nghiệm nhanh. Môi trường chấm điểm chính thức của cuộc thi là NVIDIA H200, Ubuntu 22.04, CUDA 12.x. Cell **1b** cho chọn 1 trong 3 kịch bản CUDA (dev Colab / nộp BTC / test giống BTC trên Colab) — xem chi tiết trong README. Nếu đổi cấu hình CUDA giữa chừng trong cùng session, cell **2** sẽ tự cài rồi tự restart kernel; sau khi thấy kernel restart, chạy lại từ cell 1b.

In [8]:
# 1. Clone repo (đổi REPO_URL nếu bạn fork/dùng repo khác)
REPO_URL = "https://github.com/Le-Anh-Duy/vLLM-Colab-Inference.git"
REPO_DIR = "colab-setup"

import os
if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !cd "$REPO_DIR" && git pull

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 7 (delta 5), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 2.31 KiB | 789.00 KiB/s, done.
From https://github.com/Le-Anh-Duy/vLLM-Colab-Inference
   63ad908..5a4da18  main       -> origin/main
Updating 63ad908..5a4da18
Fast-forward
 README.md                  |  9 ++++++---
 notebooks/vllm_colab.ipynb |  4 ++--
 scripts/setup_vllm.sh      | 42 ++++++++++++++++++++++++++++--------------
 3 files changed, 36 insertions(+), 19 deletions(-)


In [9]:
# 1b. Chọn kịch bản CUDA (chỉ set 1 trong 3, để mặc định = kịch bản A)
#
# A. Dev tren Colab (mac dinh, khong can set gi):
#     PINNED_CUDA_WHEEL = "0"  (dung PLAIN_CUDA_VERSION mac dinh = 13.0)
#
# B. Nop bai cho BTC (CUDA/driver da co san tren may cham, khong tu apt install):
#     PINNED_CUDA_WHEEL = "1"; CUDA_VERSION = "12.9"; SKIP_CUDA = "1"
#
# C. Test tren Colab nhung gia lap dung CUDA 12.x nhu BTC:
#     PINNED_CUDA_WHEEL = "1"; CUDA_VERSION = "12.9"
#
# LUU Y: PINNED_CUDA_WHEEL=0 dung bien PLAIN_CUDA_VERSION (mac dinh 13.0),
# PINNED_CUDA_WHEEL=1 dung bien CUDA_VERSION - hai bien nay TACH BIET, doi
# nham bien se khong co tac dung (xem comment trong scripts/setup_vllm.sh).
import os
os.environ.setdefault("PINNED_CUDA_WHEEL", "0")
# Vi du doi sang kich ban C: bo comment 2 dong duoi
# os.environ["PINNED_CUDA_WHEEL"] = "1"
# os.environ["CUDA_VERSION"] = "12.9"

'0'

In [10]:
# 2. Chạy setup (cài CUDA runtime + torch + vllm), tự phát hiện khi cấu hình
# CUDA_VERSION/VLLM_VERSION đổi so với lần chạy trước trong CÙNG session, và
# nếu có đổi thì tự cài xong rồi tự kill kernel để Colab restart (giữ nguyên
# VM/ổ đĩa, chỉ khởi động lại process Python). Sau khi thấy kernel restart,
# CHẠY LẠI CELL NÀY MỘT LẦN NỮA để tiếp tục — lần này sẽ thấy marker khớp và
# bỏ qua cài đặt.
import hashlib
import os
import subprocess
import sys

MARKER_FILE = "/content/.vllm_setup_marker"
tracked_vars = [
    "PINNED_CUDA_WHEEL",
    "CUDA_VERSION",
    "PLAIN_CUDA_VERSION",
    "SKIP_CUDA",
    "SKIP_PYTHON_DEPS",
    "VLLM_VERSION",
]
signature = hashlib.sha256(
    "|".join(f"{k}={os.environ.get(k, '')}" for k in tracked_vars).encode()
).hexdigest()

previous_signature = None
if os.path.exists(MARKER_FILE):
    with open(MARKER_FILE) as f:
        previous_signature = f.read().strip()

if previous_signature == signature:
    print("==> Cau hinh khong doi so voi lan chay truoc, bo qua setup.")
else:
    changed_mid_session = previous_signature is not None

    # Dung Popen + doc stdout theo dong va print() truc tiep, vi subprocess
    # ke thua fd goc (subprocess.call mac dinh) khong luon hien ra trong cell
    # Colab - chi thay traceback ma khong thay log that su xay ra o dau.
    proc = subprocess.Popen(
        ["bash", f"{REPO_DIR}/scripts/setup_vllm.sh"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_lines = []
    for line in proc.stdout:
        print(line, end="")
        output_lines.append(line)
    ret = proc.wait()

    if ret != 0:
        tail = "".join(output_lines[-40:])
        raise RuntimeError(
            f"setup_vllm.sh that bai voi exit code {ret}.\n"
            f"--- 40 dong log cuoi ---\n{tail}"
        )

    with open(MARKER_FILE, "w") as f:
        f.write(signature)

    if changed_mid_session:
        print("==> Cau hinh CUDA vua doi trong session nay, dang restart kernel...")
        print("==> Sau khi kernel restart xong, CHAY LAI TU CELL 1b DE TIEP TUC.")
        os.kill(os.getpid(), 9)

==> PINNED_CUDA_WHEEL=0 | CUDA_VERSION=13.0 | VLLM_VERSION=0.24.0 | SKIP_CUDA=0
1. DANG CAU HINH CUDA 13.0 RUNTIME...
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack cuda-keyring_1.1-1_all.deb ...
Unpacking cuda-keyring (1.1-1) over (1.1-1) ...
Setting up cuda-keyring (1.1-1) ...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  cuda-toolkit-13-0-config-common cuda-toolkit-13-config-common
The following NEW packages will be installed:
  cuda-cudart-13-0 cuda-toolkit-13-0-config-common
  cuda-toolkit-13-config-common
0 upgraded, 3 newly installed, 0 to remove and 121 not upgraded.
Need to get 198 kB of archives.
After this operation, 947 kB of additional disk space will b

In [13]:
# 3. Khởi động server ở background, log ra file để không block notebook
import os, subprocess, time

env = os.environ.copy()
env.setdefault("MODEL_NAME", "Qwen/Qwen2.5-1.5B-Instruct")
env.setdefault("PORT", "8000")

log_file = open("vllm_server.log", "w")
server_proc = subprocess.Popen(
    ["bash", f"{REPO_DIR}/scripts/serve_vllm.sh"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
)
print(f"server_proc.pid = {server_proc.pid}")

server_proc.pid = 5741


In [14]:
# 4. Chờ server sẵn sàng (poll /health)
import time, urllib.request, urllib.error

PORT = int(env.get("PORT", "8000"))
url = f"http://localhost:{PORT}/health"

for attempt in range(60):
    try:
        with urllib.request.urlopen(url, timeout=3) as resp:
            if resp.status == 200:
                print("Server san sang.")
                break
    except (urllib.error.URLError, ConnectionError):
        pass
    time.sleep(5)
else:
    print("Server chua san sang, xem log:")
    !tail -n 50 vllm_server.log

Server san sang.


In [15]:
# 5. Test bằng OpenAI client (vllm expose API tương thích OpenAI).
# Model la instruct nen dung chat.completions, khong dung completions tho.
!pip install -q openai

from openai import OpenAI

client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="not-needed")
response = client.chat.completions.create(
    model=env["MODEL_NAME"],
    messages=[{"role": "user", "content": "Xin chao, ban co the gioi thieu ban than khong?"}],
    max_tokens=128,
)
print(response.choices[0].message.content)

Xin chào! Tôi là trợ lý AI được tạo ra bởi Alibaba Cloud. Tôi không có mặt trên thế giới vật lý và không thể trò chuyện trực tiếp với bạn. Tuy nhiên, tôi có thể giúp bạn trả lời câu hỏi hoặc thực hiện các tác vụ dựa trên dữ liệu mà tôi đã học được. Bạn cần hỗ trợ gì cụ thể?


In [16]:
# 5b. Mo phong traffic + do TTFT/TPOT/ERS (scripts/benchmark_traffic.py).
# Nguong Floor/Ceiling/w/gamma la PLACEHOLDER, sua qua flag khi BTC cong bo
# gia tri chinh thuc tung vong (xem README).
!pip install -q httpx
!python3 "{REPO_DIR}/scripts/benchmark_traffic.py" \
    --base-url "http://localhost:{PORT}" \
    --model "{env['MODEL_NAME']}" \
    --synthetic --rps 2 --duration 30 \
    --output-dir benchmark_results

==> 64 request se duoc gui toi http://localhost:8000 (model=Qwen/Qwen2.5-1.5B-Instruct)
So request           : 64
Thanh cong / That bai : 64 / 0
TTFT (mean/p50/p95)  : 0.055s / 0.054s / 0.068s
TPOT (mean/p50/p95)  : 0.016s / 0.016s / 0.017s
ERS (uoc luong)      : 1.0000
(Nguong Floor/Ceiling/w/gamma la PLACEHOLDER, doi qua CLI flag khi BTC cong bo gia tri chinh thuc)
==> Da ghi ket qua vao benchmark_results/summary.json va benchmark_results/per_request.csv


In [17]:
# 6. Dừng server khi xong việc
server_proc.terminate()
server_proc.wait()
print("Da dung server.")

Da dung server.
